# Wasserstein Distortion example (PyTorch)

This notebook creates an adversarial attack on VGG-16 Wasserstein Distortion: starting
from random noise, it optimizes an image to minimize the distortion to a reference image.
With a large sigma, the distortion only measures local texture statistics, so the result
should look like a texture resynthesis of the reference.

In [ ]:
# Copyright 2025 Wasserstein Distortion authors.
#
# Licensed under the Apache License, Version 2.0 (the "License"); you may not use this
# file except in compliance with the License. You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software distributed under
# the License is distributed on an "AS IS" BASIS, WITHOUT WARRANTIES OR CONDITIONS OF ANY
# KIND, either express or implied. See the License for the specific language governing
# permissions and limitations under the License.

import mediapy as media
import torch
from wasserstein_distortion.torch import VGG16WassersteinDistortion

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
def to_tensor(image):
    """Converts an image array of shape (height, width, 3) to a tensor (1, 3, height, width)."""
    return torch.tensor(image[..., :3], dtype=torch.float32).permute(2, 0, 1)[None]


def to_image(x):
    """Converts a tensor of shape (1, 3, height, width) to an image array."""
    return torch.clamp(x, 0, 1)[0].permute(1, 2, 0).detach().cpu().numpy()


im1 = to_tensor(media.read_image("./example/example.png") / 255).to(device)
media.show_image(to_image(im1), title="reference")

In [ ]:
im2 = torch.nn.Parameter(torch.randn_like(im1))
optimizer = torch.optim.Adam([im2], lr=0.1)
wloss = VGG16WassersteinDistortion().to(device)
# To test the Wasserstein distortion, we construct a manual log2_sigma map
# with globally log2_sigma = 4
log2_sigma = torch.full_like(im1[:, 0:1], 4.0)

for i in range(200):
    optimizer.zero_grad()
    loss = wloss(im1, im2, log2_sigma)
    if i % 20 == 0:
        print(f"step {i}: loss = {loss.item()}")
    loss.backward()
    optimizer.step()

In [ ]:
media.show_images(
    {
        "reference": to_image(im1),
        "optimized": to_image(im2),
        "difference": to_image((im1 - im2) / 2 + 0.5),
    }
)